## Bitstream Generation

After training the Neural Network with QKeras, then to convert it to `hls4ml` and create an IP. That IP can be interfaced into a larger design to deploy on an FPGA device. In this section, we introduce the `VivadoAccelerator` backend of `hls4ml`, to deploy the model on a [Kria KV260 board].

In [1]:
# QKeras – quantization-aware training
from qkeras.qconvolutional import QConv2D, QActivation
from qkeras.utils import _add_supported_quantized_objects

from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model

from tensorflow.keras.layers import (InputLayer, 
                                     BatchNormalization,
                                     MaxPooling2D,
                                     Flatten,
                                     Dropout                                
)

co = {}
_add_supported_quantized_objects(co)
import os
import numpy as np
import tensorflow as tf

#source /tools/Xilinx/Vivado/2024.1/settings64.sh

# ── FPGA target (Kria KV260-compatible) ───────────────────────────────────
FPGA_PART    = 'xck26-sfvc784-2LV-c'
TARGET_SNR   = '76dB'

os.environ['XILINX_VIVADO'] = '/tools/Xilinx/Vivado/2024.1'
os.environ['XILINX_HLS'] = '/tools/Xilinx/Vitis_HLS/2024.1'
os.environ['PATH'] += ':/tools/Xilinx/Vitis_HLS/2024.1/bin'
os.environ['PATH'] = os.environ['XILINX_VIVADO'] + '/bin:' + os.environ['PATH']

2026-03-31 19:14:37.736029: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Load model

Load the quantized trained model.

Define path for vivado and follow instructions for bitstream generation

In [ ]:
# ── Datasets parameters ─────────────────────────────────────────────
DATASETS_PATH = "MFCC_datasets"
METADATA_PATH = os.path.join(DATASETS_PATH, "metadata.json")

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
    MFCC_input_shape = tuple(metadata["MFCC_input_shape"])
    NUM_CLASSES      = metadata["num_classes"]
    N_MFCC           = metadata.get("N_MFCC", 20)
    MFCC_FRAMES      = metadata.get("MFCC_FRAMES")
    SAMPLING_FREQ    = metadata.get("SAMPLING_FREQ", 48000)
    batch_size       = metadata.get("batch_size", 32)
    subfolders       = metadata.get("subfolders", [])
    unique_labels    = metadata.get("labels", [])
    print("Metadata loaded from metadata.json")

#    (16, 6, 30, 1e-5), # 16-bit: fine-tuning suave desde float
#    ( 8, 4, 40, 1e-4), # 8-bit:  paso min ~0.0625, LR 1e-4 → ~62 épocas/step
#    ( 4, 2, 120, 5e-4), # 4-bit:  paso min ~0.25,   LR 5e-4 → ~50 épocas/stepbits = 8 #Select 32, 16, 8 or 4-bit. 32 is for float model
bits = 8 
int_bits = 4
ft_epochs = 40
ft_lr = 1e-4

label     = f'{bits}bit'
ckpt_dir  = f'./ckpt_MFCC_QAT_{label}_{TARGET_SNR}'
#qmodel_path = f'ckpt_MFCC_QAT_{label}_76dB/MFCC_QAT_{label}_best.h5'
qmodel_path = f'{ckpt_dir}/compact_QAT_{label}_best.h5'
ip_path = f'./hls4ml_output/qat_{label}/export_cnn_ip'

print(f"Importing h5 model from: {qmodel_path}")

model = load_model(qmodel_path, custom_objects=co)
#!sed -n '30,45p' model_3/hls4ml_prj_pynq/myproject_vivado_accelerator/project_1.runs/impl_1/design_1_wrapper_utilization_placed.rpt

Importing h5 model from: ./ckpt_MFCC_QAT_8bit_76dB/compact_QAT_8bit_best.h5


## Convert to hls4ml

In [ ]:
import hls4ml
#import plotting

config = hls4ml.utils.config_from_keras_model(model, granularity='name')

# Set reuse factor for layers
layer_rf = {
    'conv1':  1,    # 144 MACs / 1 = 144 DSPs
    'conv2':  4,    # 576 MACs / 4 = 144 DSPs
    'conv3':  4,    # 576 MACs / 4 = 144 DSPs
    'dense1': 1,    # 256 MACs / 1 = 256 DSPs
    'output': 1,    # 160 MACs / 1 = 160 DSPs
}
for lname, rf in layer_rf.items():
    if lname in config.get('LayerName', {}):
        config['LayerName'][lname]['ReuseFactor'] = rf
        config['LayerName'][lname]['Strategy']    = 'Latency'

print("-----------------------------------")


#plotting.print_dict(config)

#hls_model = hls4ml.converters.convert_from_keras_model(
#    model, hls_config=config, output_dir=ip_path, backend='VivadoAccelerator', board='pynq-z2'
#)

#print(f"Saving output files at: {ip_path}")

#hls_model.compile()

/home/barnesrob/miniforge3/envs/hls4ml_env/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


-----------------------------------
Saving output files at: ./hls4ml_output/qat_8bit/export_cnn_ip


In [ ]:

# Custom-objects dict needed when reloading QKeras models
QKERAS_CUSTOM_OBJECTS = {}
_add_supported_quantized_objects(QKERAS_CUSTOM_OBJECTS)


os.makedirs(ckpt_dir, exist_ok=True)

print(f"\n{'='*60}")
print(f'  QAT {bits}-bit  (integer_bits={int_bits})')
print(f"{'='*60}")

if os.path.exists(qmodel_path):
    print(f'  Pre-trained QAT model found: {qmodel_path}')
    print('  Loading.  Delete this file to retrain.')
    model_q = tf.keras.models.load_model(
        qmodel_path, custom_objects=QKERAS_CUSTOM_OBJECTS)
    
    # FIX #7: always recompile with explicit parameters
    model_q.compile(
        optimizer=Adam(learning_rate=ft_lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
else:
    print(f"Building QAT model and transferring float weights ...")
    
    model_q = build_qat_model(MFCC_input_shape, NUM_CLASSES, bits, int_bits)
    model_q.summary()

    verify_layer_sizes(model_q)

    model_q.compile(
        optimizer=Adam(learning_rate=ft_lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    transfer_float_weights(model_MFCC, model_q)

    # FIX #3: fresh callbacks per training session
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=qmodel_path, monitor='val_accuracy',
            save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1
        ),
    ]

    hist_q = model_q.fit(
        MFCC_dataset_train_batches,
        epochs=ft_epochs,
        validation_data=MFCC_dataset_validation_batches,
        callbacks=[early_stopping_callback]
    )

    # Save best model manually in H5 format (without passing 'options')
    model_q.save(qmodel_path, save_format='h5')
    print(f'Model saved → {qmodel_path}')
    save_history(hist_q, hist_path)

if os.path.exists(hist_path):
    plot_training_history(load_history(hist_path),
                            title=f'QAT {bits}-bit - MFCC {TARGET_SNR}')


The `create_initial_config` method (of any backend) creates a template dictionary with the default parameters that you can use as a starting point. In the conversion above, we didn't change any of these settings so all the defaults are used.

In [ ]:
hls4ml.backends.get_backend('VivadoAccelerator').create_initial_config()

## Take top module and add needed pragmas for Axi-stream ports
Run the CPU emulation of the hls4ml NN and save the file to compare against the hardware result later.

Build the model.

## Generate diagrams

HLS4ML diagram

In [10]:
import sys
import os

# Add PlotNeuralNet to Python path
sys.path.append('/home/barnesrob/Documents/VSProj/CNN_HLS4ML')  # Update this path
# Or if cloned in current directory:
sys.path.append('./PlotNeuralNet')


sys.path.append('./PlotNN_output')
from pycore.tikzeng import *
from pycore.blocks  import *

# defined your arch
arch = [
    to_head( '..'),#
    to_cor(),
    to_begin(),

    # Input Layer
    to_input('input.png', name='input', to='(-1,0,0)', width=8, height=8),#, caption="MFCC input"
   
    #*block_2ConvPool( name='conv1', botton='input', top='pool1', s_filer=256, n_filer=128, offset="(0,0,0)", size=(64,64,3.5), opacity=0.5 ),
    to_ConvConvRelu("conv1", s_filer=64, n_filer=(64,64), offset="(0,0,0)", to="(2,0,0)", height=32, depth=32, width=(3,3), caption="Conv1" ),
    #to_Conv("conv1", 64, 8, offset="(0,0,0)", to="(3,0,0)", height=64, depth=64, width=3, caption="Conv1" ),
    
    #to_connection("input", "conv1"),
    to_Pool("pool1", offset="(0,0,0)", to="(conv1-east)", height=24, depth=24, width=1),

    #to_Conv("conv2", 32, 8, offset="(4,0,0)", to="(pool1-east)", height=32, depth=32, width=3, caption="Conv2" ),
    to_ConvConvRelu("conv2", s_filer=32, n_filer=(32,32), offset="(2,0,0)", to="(pool1-east)", height=16, depth=16, width=(3,3), caption="Conv2" ),
    to_connection("pool1", "conv2"),
    to_Pool("pool2", offset="(0,0,0)", to="(conv2-east)", height=12, depth=12, width=1),

    #to_Conv("conv3", 8, 8, offset="(2,0,0)", to="(pool2-east)", height=8, depth=8, width=3, caption="Conv3" ),
    to_ConvConvRelu("conv3", s_filer=8, n_filer=(8,8), offset="(1.5,0,0)", to="(pool2-east)", height=10, depth=10, width=(3,3), caption="Conv3" ),
    to_connection("pool2", "conv3"),
    to_Pool("pool3", offset="(0,0,0)", to="(conv3-east)", height=6, depth=6, width=1),

    to_SoftMax("soft1", 20 ,"(2,0,0)", "(pool3-east)"  , caption=" \\\\~\\\\Spkr.\\\\ Recognition"  ),#, caption="Spkr Recognition"
    to_connection("pool3", "soft1"),

    #to_Sum("sum1", offset="(1.5,0,0)", to="(soft1-east)", radius=2.5, opacity=0.6),
    #to_connection("soft1", "sum1"),

    to_end()
    ]

def main():
    Folder = './PlotNeuralNet'
    subFol = '/PlotNN_output'
    namefile = "Network_Float"
    output_dir = f'{Folder}{subFol}/'

    # Create directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate the file
    full_path = f'{output_dir}/{namefile}'
    to_generate(arch,  full_path + '.tex' )

    print(f"Generated {namefile}.tex")

if __name__ == '__main__':
    main()




\documentclass[border=8pt, multi, tikz]{standalone} 
\usepackage{import}
\subimport{../layers/}{init}
\usetikzlibrary{positioning}
\usetikzlibrary{3d} %for including external image 


\def\ConvColor{rgb:yellow,5;red,2.5;white,5}
\def\ConvReluColor{rgb:yellow,5;red,5;white,5}
\def\PoolColor{rgb:red,1;black,0.3}
\def\UnpoolColor{rgb:blue,2;green,1;black,0.3}
\def\FcColor{rgb:blue,5;red,2.5;white,5}
\def\FcReluColor{rgb:blue,5;red,5;white,4}
\def\SoftmaxColor{rgb:magenta,5;black,7}   
\def\SumColor{rgb:blue,5;green,15}


\newcommand{\copymidarrow}{\tikz \draw[-Stealth,line width=0.8mm,draw={rgb:blue,4;red,1;green,1;black,3}] (-0.3,0) -- ++(0.3,0);}

\begin{document}
\begin{tikzpicture}
\tikzstyle{connection}=[ultra thick,every node/.style={sloped,allow upside down},draw=\edgecolor,opacity=0.7]
\tikzstyle{copyconnection}=[ultra thick,every node/.style={sloped,allow upside down},draw={rgb:blue,4;red,1;green,1;black,3},opacity=0.7]


\node[canvas is zy plane at x=0] (input) at (-1,0,0) {\in

In [2]:
## FINN FC Arch diagram

# defined your arch
arch = [
    to_head( '..'),#
    to_cor(),
    to_begin(),

    # Input Layer
    to_input('input.png', name='input', to='(-2,0,0)', width=8, height=8),#, caption="MFCC input"
   
    #*block_2ConvPool( name='conv1', botton='input', top='pool1', s_filer=256, n_filer=128, offset="(0,0,0)", size=(64,64,3.5), opacity=0.5 ),
    to_UnPool("flat", offset="(0,0,0)", to="(1,0,0)", height=32, depth=32, width=3, caption="Flatten" ),
    
    to_Conv("linear", 1280, 1280, offset="(0,0,0)", to="(4,0,0)", height=32, depth=32, width=3, caption="Linear" ),
    
    to_connection("flat", "linear"),
    #to_Pool("pool1", offset="(0,0,0)", to="(flat-east)"),

    #to_Conv("fc1", 32, 8, offset="(4,0,0)", to="(pool1-east)", height=64, depth=64, width=3, caption="FC1" ),
    to_ConvConvRelu("fc1", s_filer=256, n_filer=(256,256), offset="(2,0,0)", to="(linear-east)", height=22, depth=22, width=(3,3), caption="FC1" ),
    to_connection("linear", "fc1"),
    to_Pool("pool2", offset="(0,0,0)", to="(fc1-east)", height=12, depth=12, width=1),

    #to_Conv("conv3", 8, 8, offset="(2,0,0)", to="(pool2-east)", height=8, depth=8, width=3, caption="Conv3" ),
    to_ConvConvRelu("fc2", s_filer=64, n_filer=(64,64), offset="(1,0,0)", to="(pool2-east)", height=10, depth=10, width=(3,3), caption="FC2" ),
    to_connection("pool2", "fc2"),
    to_Pool("pool3", offset="(0,0,0)", to="(fc2-east)", height=6, depth=6, width=1),

    to_SoftMax("soft1", 10 ,"(2,0,0)", "(pool3-east)" , caption=" \\\\~\\\\Spkr.\\\\ Recognition"  ),#, caption="Spkr Recognition"
    to_connection("pool3", "soft1"),

    #to_Sum("sum1", offset="(1.5,0,0)", to="(soft1-east)", radius=2.5, opacity=0.6),
    #to_connection("soft1", "sum1"),

    to_end()
    ]

def main():
    Folder = './PlotNeuralNet'
    subFol = '/PlotNN_output'
    namefile = "Network_FC_FINN"
    output_dir = f'{Folder}{subFol}/'

    # Create directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate the file
    full_path = f'{output_dir}/{namefile}'
    to_generate(arch,  full_path + '.tex' )

    print(f"Generated {namefile}.tex")

if __name__ == '__main__':
    main()




\documentclass[border=8pt, multi, tikz]{standalone} 
\usepackage{import}
\subimport{../layers/}{init}
\usetikzlibrary{positioning}
\usetikzlibrary{3d} %for including external image 


\def\ConvColor{rgb:yellow,5;red,2.5;white,5}
\def\ConvReluColor{rgb:yellow,5;red,5;white,5}
\def\PoolColor{rgb:red,1;black,0.3}
\def\UnpoolColor{rgb:blue,2;green,1;black,0.3}
\def\FcColor{rgb:blue,5;red,2.5;white,5}
\def\FcReluColor{rgb:blue,5;red,5;white,4}
\def\SoftmaxColor{rgb:magenta,5;black,7}   
\def\SumColor{rgb:blue,5;green,15}


\newcommand{\copymidarrow}{\tikz \draw[-Stealth,line width=0.8mm,draw={rgb:blue,4;red,1;green,1;black,3}] (-0.3,0) -- ++(0.3,0);}

\begin{document}
\begin{tikzpicture}
\tikzstyle{connection}=[ultra thick,every node/.style={sloped,allow upside down},draw=\edgecolor,opacity=0.7]
\tikzstyle{copyconnection}=[ultra thick,every node/.style={sloped,allow upside down},draw={rgb:blue,4;red,1;green,1;black,3},opacity=0.7]


\node[canvas is zy plane at x=0] (input) at (-2,0,0) {\in

In [32]:
## FINN BNN-PYNQ Arch diagram

# defined your arch
arch = [
    to_head( '..'),#
    to_cor(),
    to_begin(),

    # Input Layer
    to_input('input.png', name='input', to='(-1,0,0)', width=7, height=7),#, caption="MFCC input"
   
    to_Conv("conv1", 8, 16, offset="(0,0,0)", to="(2,0,0)", height=32, depth=32, width=3, caption="Conv2D" ),
    #to_ConvConvRelu("conv1", s_filer=32, n_filer=(32,32), offset="(0,0,0)", to="(2,0,0)", height=32, depth=32, width=(3,3), caption="Conv1" ),
    to_ConvConvRelu("bn1", s_filer=16, n_filer=(16,16), offset="(1,0,0)", to="(4,0,0)", height=32, depth=32, width=(3,3), caption=" \\\\~\\\\~\\\\~BN+ReLu\\\\+\\\\MaxPool" ),
    
    to_connection("conv1", "bn1"),
    to_Pool("pool2", offset="(0,0,0)", to="(bn1-east)", height=16, depth=16, width=1),

    to_Pool("pool3", offset="(2,0,0)", to="(6,0,0)", height=16, depth=16, width=1,caption="MaxPool"),
    to_connection("pool2", "pool3"),
    to_Pool("pool4", offset="(0,0,0)", to="(pool3-east)", height=8, depth=16, width=1),

    #to_Conv("conv3", 8, 8, offset="(2,0,0)", to="(pool2-east)", height=8, depth=8, width=3, caption="Conv3" ),
    #to_ConvConvRelu("conv2", s_filer=64, n_filer=(64,64), offset="(2,0,0)", to="(pool2-east)", height=16, depth=16, width=(3,3), caption="Conv2" ),
    #to_ConvConvRelu("bn2", s_filer=64, n_filer=(64,64), offset="(0.5,0,0)", to="(conv2-east)", height=16, depth=16, width=(3,3), caption=" \\\\~\\\\~\\\\~\\\\~\\\\BN\\\\+\\\\ReLu\\\\+\\\\Pool" ),
    #to_connection("pool2", "conv2"),

    to_UnPool("flat1", offset="(0,0,0)", to="(11,0,0)", height=8, depth=16, width=3, caption=" \\\\~\\\\Flatten\\\\+\\\\FC" ),
    to_connection("pool4", "flat1"),
    to_Conv("flat2", offset="(2,0,0)", to="(flat1-east)", height=3, depth=16, width=3, caption=" \\\\~\\\\~\\\\~\\\\ReLu\\\\+\\\\FC" ),
    to_connection("flat1", "flat2"),

    to_SoftMax("soft1", 10 ,"(2,0,0)", "(flat2-east)" , caption=" \\\\~\\\\Spkr.\\\\ Recognition" ),#, caption="Spkr Recognition"
    to_connection("flat2", "soft1"),

    #to_Sum("sum1", offset="(1.5,0,0)", to="(soft1-east)", radius=2.5, opacity=0.6),
    #to_connection("soft1", "sum1"),

    to_end()
    ]

def main():
    Folder = './PlotNeuralNet'
    subFol = '/PlotNN_output'
    namefile = "Network_BNN-PYNQ_FINN"
    output_dir = f'{Folder}{subFol}/'

    # Create directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate the file
    full_path = f'{output_dir}/{namefile}'
    to_generate(arch,  full_path + '.tex' )

    print(f"Generated {namefile}.tex")

if __name__ == '__main__':
    main()




\documentclass[border=8pt, multi, tikz]{standalone} 
\usepackage{import}
\subimport{../layers/}{init}
\usetikzlibrary{positioning}
\usetikzlibrary{3d} %for including external image 


\def\ConvColor{rgb:yellow,5;red,2.5;white,5}
\def\ConvReluColor{rgb:yellow,5;red,5;white,5}
\def\PoolColor{rgb:red,1;black,0.3}
\def\UnpoolColor{rgb:blue,2;green,1;black,0.3}
\def\FcColor{rgb:blue,5;red,2.5;white,5}
\def\FcReluColor{rgb:blue,5;red,5;white,4}
\def\SoftmaxColor{rgb:magenta,5;black,7}   
\def\SumColor{rgb:blue,5;green,15}


\newcommand{\copymidarrow}{\tikz \draw[-Stealth,line width=0.8mm,draw={rgb:blue,4;red,1;green,1;black,3}] (-0.3,0) -- ++(0.3,0);}

\begin{document}
\begin{tikzpicture}
\tikzstyle{connection}=[ultra thick,every node/.style={sloped,allow upside down},draw=\edgecolor,opacity=0.7]
\tikzstyle{copyconnection}=[ultra thick,every node/.style={sloped,allow upside down},draw={rgb:blue,4;red,1;green,1;black,3},opacity=0.7]


\node[canvas is zy plane at x=0] (input) at (-1,0,0) {\in

# Make font Large

% Add these lines to change font size
\usepackage{relsize}
\tikzset{every node/.append style={font=\Large}}  % Options: \large, \Large, \LARGE, \huge, \Huge

# Compile the .tex files in terminal:

pdflatex Network_BNN-PYNQ_FINN.tex 